# Notebook 2 — Entraînement et évaluation externe

**Réglages Kaggle :**
- *Accelerator* : **GPU T4** (évitez le P100 : les versions récentes de PyTorch ne le prennent plus en charge)
- *Internet* : **On** (téléchargement de ClinicalBERT)
- *Add Input* : le dataset du code **et** le dataset `ecg-fusion-cache` (sortie du notebook 1)

Une session Kaggle est limitée à 12 h. L'entraînement s'arrête proprement à `MAX_HOURS`
et sauvegarde `last.pt`. **Pour continuer, il n'y a rien à modifier dans ce notebook** :
1. Ouvrez ce notebook en mode édition (*Edit*).
2. *Add Input* → *Your Work* → choisissez **ce même notebook** (sa dernière version).
3. *Save Version → Save & Run All*.

Le notebook détecte tout seul `last.pt` et reprend à l'époque suivante. Répétez jusqu'à voir
**ENTRAÎNEMENT TERMINÉ** ; l'évaluation externe se lance alors automatiquement.

In [ ]:
import glob, os, re, shutil
def _version(init):
    m = re.search(r'__version__\s*=\s*"([^"]+)"', open(init).read())
    return m.group(1) if m else "0"
# plusieurs copies du code peuvent être présentes (dataset du code, sorties d'anciens notebooks) :
# on prend la plus récente
hits = sorted(glob.glob("/kaggle/input/**/ecgfusion/__init__.py", recursive=True), key=_version)
assert hits, "Code introuvable : ajoutez le dataset contenant ecg_fusion.zip via « Add Input »."
src = os.path.dirname(os.path.dirname(hits[-1]))
shutil.rmtree("/tmp/ecg_fusion", ignore_errors=True)
shutil.copytree(src, "/tmp/ecg_fusion")   # hors /kaggle/working : le code n'encombre pas la sortie
os.chdir("/tmp/ecg_fusion")
!pip install -q wfdb
print(f"Code version {_version(hits[-1])} prêt (source : {src})")

In [ ]:
import json
meta = glob.glob("/kaggle/input/**/cache/mimic/meta.csv", recursive=True)
assert meta, "Cache introuvable : ajoutez la sortie du notebook 1 comme Input."
CACHE = os.path.dirname(os.path.dirname(meta[0]))
_i = json.load(open(f"{CACHE}/mimic/info.json"))
if not _i.get("complete", True):
    print(f"ATTENTION : cache MIMIC incomplet ({_i['n_done']}/{_i['n']}). Terminez le notebook 1 avant d'entraîner.")
CONFIG = "configs/kaggle_v2.yaml"   # v1 : configs/kaggle.yaml
RUN_NAME = "fusion_v2"              # changer de nom pour chaque nouvelle expérience
RUN = f"/kaggle/working/runs/{RUN_NAME}"
MAX_HOURS = 10      # marge pour la calibration et la sauvegarde avant la limite des 12 h
# Copie des caches sur le disque local de la machine. Lire des ECG au hasard directement dans
# /kaggle/input est extrêmement lent (le 1er test : ~25 min pour 12 000 ECG) ; une copie
# séquentielle prend quelques minutes, ensuite chaque époque est rapide.
import time
LOCAL = "/tmp/cache_local"          # hors /kaggle/working : non sauvegardé, recréé à chaque session
def copy_cache(name):
    src, dst = f"{CACHE}/{name}", f"{LOCAL}/{name}"
    os.makedirs(dst, exist_ok=True)
    t0, total = time.time(), 0
    for f in sorted(os.listdir(src)):
        s_, d_ = os.path.join(src, f), os.path.join(dst, f)
        if os.path.exists(d_) and os.path.getsize(d_) == os.path.getsize(s_):
            continue
        shutil.copyfile(s_, d_); total += os.path.getsize(d_)
    dt = time.time() - t0
    print(f"  {name:6s} {total / 1e9:5.1f} Go copiés en {dt / 60:4.1f} min ({total / 1e6 / max(dt, 1e-6):.0f} Mo/s)")
need = sum(os.path.getsize(f"{CACHE}/{d}/signals.npy") for d in ["mimic", "ptbxl", "lsas"])
free = shutil.disk_usage("/tmp").free
print(f"Disque local libre : {free / 1e9:.0f} Go, besoin : {need / 1e9:.1f} Go")
if free > 1.15 * need:
    for d in ["mimic", "ptbxl", "lsas"]:
        copy_cache(d)
    SRC = LOCAL
else:
    print("ATTENTION : pas assez de place pour la copie locale ; lecture directe (lente) depuis l'Input")
    SRC = CACHE
# Labels MIMIC recalculés avec les règles actuelles du code (quelques secondes)
MIMIC = "/tmp/cache_labels/mimic"
!python scripts/audit_labels.py --cache {SRC}/mimic --relabel --link --quiet --out {MIMIC}
PATHS = (f"data.train_cache={MIMIC} "
         f"data.eval_sets.mimic_internal.cache={MIMIC} "
         f"data.eval_sets.mimic_val.cache={MIMIC} data.eval_sets.mimic_val.split=val "
         f"data.eval_sets.ptbxl_external.cache={SRC}/ptbxl "
         f"data.eval_sets.lsas_external.cache={SRC}/lsas")

# Reprise automatique : cherche un last.pt dans les Inputs (sortie d'une version précédente)
prev = sorted(glob.glob(f"/kaggle/input/**/runs/{RUN_NAME}/last.pt", recursive=True))
PREV = os.path.dirname(prev[0]) if prev else None
print("Cache :", CACHE)
if PREV:
    st = json.load(open(f"{PREV}/status.json")) if os.path.exists(f"{PREV}/status.json") else {}
    print(f"REPRISE depuis {PREV} (dernière époque : {st.get('last_epoch', '?')}, "
          f"meilleur val : {st.get('best_val', float('nan')):.4f})")
else:
    print("NOUVEL ENTRAÎNEMENT (aucun last.pt trouvé dans les Inputs)")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Test rapide (désactivé par défaut : `RUN_TEST = True` pour le relancer)

In [ ]:
RUN_TEST = False   # mettre True pour refaire un essai de 1 époque sur 2 000 ECG
if PREV is None and RUN_TEST:
    !python scripts/train.py --config {CONFIG} --set {PATHS} output_dir=/kaggle/working/runs/test train.epochs=1 data.max_train_records=2000
else:
    print("Test ignoré")

## Entraînement (nouveau ou reprise, automatique)

In [ ]:
RESUME = f"--resume {PREV}" if PREV else ""
!python scripts/train.py --config {CONFIG} --max_hours {MAX_HOURS} {RESUME} --set {PATHS} output_dir={RUN}
STATUS = json.load(open(f"{RUN}/status.json"))
print("=" * 60)
if STATUS["finished"]:
    print(f"ENTRAÎNEMENT TERMINÉ ({STATUS['stop_reason']}) - époque {STATUS['last_epoch']}, "
          f"meilleur val = {STATUS['best_val']:.4f}")
else:
    print(f"À REPRENDRE : arrêt à l'époque {STATUS['last_epoch']} (budget de temps). "
          "Ajoutez la sortie de cette version en Input puis relancez.")
print("=" * 60)

## Évaluation interne + externe (seulement quand l'entraînement est terminé)

In [ ]:
if STATUS["finished"]:
    !python scripts/evaluate.py --run {RUN} --set {PATHS}
else:
    print("Entraînement non terminé : évaluation reportée à la prochaine version.")

## Analyse a posteriori (CPU, quelques minutes) : prévalences, calibration, gating, DeLong

In [ ]:
if STATUS["finished"]:
    !python scripts/analyze_preds.py --run {RUN}

## Résultats

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
if not STATUS["finished"]:
    print("Pas encore de résultats : entraînement à reprendre (voir l'en-tête du notebook).")
else:
    s = pd.read_csv(f"{RUN}/eval/summary.csv")
    display(s[["eval_set", "scenario", "n", "macro_auroc", "macro_auroc_ci_low", "macro_auroc_ci_high",
               "macro_f1", "macro_ece", "rare_macro_auroc"]].round(3))
    pl = pd.read_csv(f"{RUN}/eval/per_label.csv")
    display(pl[pl.scenario == "no_text"].pivot(index="label", columns="eval_set", values="auroc").round(3))
    display(pd.read_csv(f"{RUN}/eval/modality_contribution.csv").round(3))

In [ ]:
# courbe d'apprentissage (toutes les sessions cumulées)
h = pd.read_csv(f"{RUN}/history.csv")
ax = h.plot(x="epoch", y=[c for c in h.columns if c.startswith("val_auroc")], marker="o", figsize=(7, 4))
ax.set_ylabel("AUROC macro (validation MIMIC)"); ax.grid(alpha=.3)